# Week 3 — Train an RL agent on Colab

Free Colab, no GPU needed. **Also runs on Kaggle Notebooks** (30 free GPU hrs/week) — the code is identical; only the data-clone path differs, noted below. This notebook:
1. Gets the code and data
2. Splits history into **train** (early) and **test** (later)
3. Trains PPO, **checkpointing so a disconnect never costs you your run**
4. Evaluates on the test period and plots your agent vs the benchmark

The agent is just a learned `state -> weights` function. Everything else is the
env you've used since week 1.

## 1. Get the code and data
Clones the repo and installs. Re-run safely; it skips what's already there.

In [ ]:
import os
if not os.path.exists('efg-algo-internship'):
    !git clone https://github.com/AhmedRajaie/efg-algo-internship.git
%cd efg-algo-internship
!pip -q install "stable-baselines3>=2.7,<3" "gymnasium>=1.0,<2" >/dev/null
import sys; sys.path.insert(0, 'src')
print('ready')

## 2. (Optional) Mount Drive for checkpoints
Colab disconnects when idle. Saving checkpoints to Drive means you can **resume**
instead of starting over. Skip this cell to keep checkpoints only in the session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/efg_rl_checkpoints'
import os; os.makedirs(CKPT_DIR, exist_ok=True)
print('checkpoints ->', CKPT_DIR)

## 3. Load the data and split it
Train on the early years, test on the later ones. The agent never sees test during training.

In [ ]:
from tradinglab.data_feed import DataFeed
from tradinglab.train import split_day, make_envs

feed = DataFeed.from_dir('data/egx')
split = split_day(feed, train_frac=0.7)
print('universe:', feed.symbols)
print('total days:', feed.n_days, '| train up to day', split, '| test from day', split)

train_env, test_env = make_envs(feed, split, reward='excess', top_k=2, lookback=30)

## 4. Train with checkpointing
`CheckpointCallback` saves every `save_freq` steps. If Colab drops, re-run the
resume cell below to continue from the last checkpoint instead of from zero.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import CheckpointCallback

CKPT_DIR = globals().get('CKPT_DIR', './checkpoints')
import os; os.makedirs(CKPT_DIR, exist_ok=True)

ckpt = CheckpointCallback(save_freq=5000, save_path=CKPT_DIR, name_prefix='ppo')
model = PPO('MlpPolicy', train_env, verbose=1, n_steps=512, seed=0)
model.learn(total_timesteps=100_000, callback=ckpt)
model.save(os.path.join(CKPT_DIR, 'ppo_final'))
print('done')

### Resume after a disconnect
Run this **instead of** the cell above if you got dropped mid-training.

In [ ]:
import glob, os
from stable_baselines3 import PPO
ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, 'ppo_*_steps.zip')))
assert ckpts, 'no checkpoint found — train from scratch above'
model = PPO.load(ckpts[-1], env=train_env)
print('resumed from', ckpts[-1])
model.learn(total_timesteps=50_000, reset_num_timesteps=False)
model.save(os.path.join(CKPT_DIR, 'ppo_final'))

## 5. Evaluate on TEST and plot
The test curve is the only number that counts. Train-period gains can be memorized noise.

In [ ]:
import matplotlib.pyplot as plt
from tradinglab.train import evaluate

tr = evaluate(model, train_env)
te = evaluate(model, test_env)
print('TRAIN  agent %.3f  benchmark %.3f' % (tr['final_portfolio'], tr['final_benchmark']))
print('TEST   agent %.3f  benchmark %.3f' % (te['final_portfolio'], te['final_benchmark']))

plt.figure(figsize=(10,5))
plt.plot(te['portfolio'], label='agent (test)')
plt.plot(te['benchmark'], label='benchmark (test)')
plt.title('Agent vs benchmark — TEST period'); plt.legend(); plt.grid(alpha=.3)
plt.show()

## 6. Experiment
Change one thing, retrain, compare. Ideas:
- `reward='return'` vs `'excess'` vs `'risk_adjusted'` — watch behavior change
- `top_k` — how many stocks the agent holds
- `commission=0.001` — does it still win after costs?
- more `total_timesteps` — does test performance improve, or just train?